[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/04.%20The%20FCFS%20Berth%20Scheduling%20Problem/P4-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 4. The FCFS Berth Scheduling Problem

## Tier 3: PSO for FCFS-Constrained Berth Scheduling (Advanced Algorithm)

### Key assumptions
- Particles represent complete berth scheduling solutions in continuous search space
- FCFS constraints are maintained through specialized position decoding procedures
- Swarm intelligence explores solution space while preserving feasibility
- Physical berth constraints (length, draft) are enforced during decoding
- Processing times are deterministic and vessels must maintain arrival order precedence

### Approach (step-by-step)
1. **Particle encoding**: Each particle position vector contains berth preferences for each vessel
2. **Swarm initialization**: Random particle positions within feasible search space
3. **FCFS decoding**: Convert continuous positions to discrete berth-time assignments
4. **Fitness evaluation**: Calculate total waiting time and makespan for each particle
5. **PSO updates**: Apply velocity and position updates with inertia and social/cognitive components
6. **Constraint handling**: Ensure FCFS compliance during solution decoding
7. **Convergence monitoring**: Track best solutions and swarm diversity

### What to look for in the results
- Superior performance vs both mathematical optimization and heuristic approaches
- Convergence behavior showing swarm learning and improvement over iterations
- Balance between exploration (diversity) and exploitation (best solution refinement)
- FCFS constraint satisfaction while optimizing berth assignments
- Reduced waiting times and improved berth utilization

### Concrete example (from the source)
Advanced scenario with 6 vessels and 4 berths demonstrating PSO optimization:
- Vessel 1: Arrival=0h, Processing=6h, Priority=High
- Vessel 2: Arrival=1h, Processing=4h, Priority=Medium
- Vessel 3: Arrival=2h, Processing=8h, Priority=High
- Vessel 4: Arrival=2.5h, Processing=3h, Priority=Low
- Vessel 5: Arrival=4h, Processing=5h, Priority=Medium
- Vessel 6: Arrival=4h, Processing=7h, Priority=High

PSO Parameters:
- Swarm size: 30 particles
- Inertia weight: 0.7 (balance exploration/exploitation)
- Cognitive coefficient: 1.5 (individual learning)
- Social coefficient: 1.5 (swarm learning)
- Max iterations: 100

Expected PSO performance improvements:
- 47% reduction in total waiting time vs priority-based FCFS
- Improved berth utilization (85.7% vs 76.7%)
- Faster convergence (45 iterations to reach near-optimal solution)
- Maintained FCFS compliance with optimized berth assignments

In [ ]:
# Import required packages for PSO algorithm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import itertools
from collections import defaultdict
import random
import time
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All packages imported successfully for PSO Algorithm")

In [ ]:
@dataclass
class PSOVessel:
    """Vessel representation for PSO optimization"""
    id: int
    arrival_time: float
    processing_time: float
    priority_score: float  # For fitness evaluation
    max_length: float = 400
    max_draft: float = 16

@dataclass
class PSOBerth:
    """Berth representation for PSO optimization"""
    id: int
    max_length: float
    max_draft: float

@dataclass
class Particle:
    """PSO particle representing a complete scheduling solution"""
    position: np.ndarray  # Continuous berth preferences for each vessel
    velocity: np.ndarray  # Velocity vector for position updates
    fitness: float        # Fitness value (total waiting time + makespan)
    best_position: np.ndarray  # Personal best position
    best_fitness: float      # Personal best fitness
    schedule: List[Dict]     # Decoded schedule assignment

@dataclass
class PSOSolution:
    """Complete PSO solution with schedule and metrics"""
    schedule: List[Dict]
    total_waiting_time: float
    makespan: float
    berth_utilization: Dict[int, float]
    fcfs_compliance: float
    convergence_iterations: int
    final_fitness: float

print("✅ PSO data structures defined successfully")

In [ ]:
class PSOFCFSScheduler:
    """Particle Swarm Optimization for FCFS-Constrained Berth Scheduling"""
    
    def __init__(self, vessels: List[PSOVessel], berths: List[PSOBerth],
                 swarm_size: int = 30, max_iterations: int = 100,
                 inertia_weight: float = 0.7, cognitive_coeff: float = 1.5,
                 social_coeff: float = 1.5, random_seed: Optional[int] = None):
        """
        Initialize PSO scheduler for FCFS berth scheduling
        
        Args:
            vessels: List of vessels to be scheduled
            berths: List of available berths
            swarm_size: Number of particles in the swarm
            max_iterations: Maximum number of PSO iterations
            inertia_weight: Inertia weight for velocity updates
            cognitive_coeff: Cognitive learning coefficient
            social_coeff: Social learning coefficient
            random_seed: Random seed for reproducibility
        """
        self.vessels = vessels
        self.berths = berths
        self.num_vessels = len(vessels)
        self.num_berths = len(berths)
        
        # PSO parameters
        self.swarm_size = swarm_size
        self.max_iterations = max_iterations
        self.inertia_weight = inertia_weight
        self.cognitive_coeff = cognitive_coeff
        self.social_coeff = social_coeff
        
        # Sort vessels by arrival time for FCFS constraint handling
        self.vessels_sorted = sorted(vessels, key=lambda v: v.arrival_time)
        
        # Initialize random seed
        if random_seed is not None:
            np.random.seed(random_seed)
            random.seed(random_seed)
        
        # Swarm and solution tracking
        self.particles = []
        self.global_best_position = None
        self.global_best_fitness = float('inf')
        self.global_best_schedule = None
        
        # Convergence tracking
        self.convergence_history = []
        self.diversity_history = []
        
        # Final solution
        self.solution = None
        
    def _initialize_swarm(self):
        """Initialize the PSO swarm with random positions and velocities"""
        print("🐝 Initializing PSO swarm...")
        
        self.particles = []
        
        for i in range(self.swarm_size):
            # Random position: berth preferences for each vessel [0, num_berths-1]
            position = np.random.uniform(0, self.num_berths - 0.01, self.num_vessels)
            
            # Random velocity: small random values
            velocity = np.random.uniform(-0.5, 0.5, self.num_vessels)
            
            # Create particle
            particle = Particle(
                position=position,
                velocity=velocity,
                fitness=float('inf'),
                best_position=position.copy(),
                best_fitness=float('inf'),
                schedule=[]
            )
            
            # Evaluate initial fitness
            self._evaluate_particle(particle)
            
            # Update personal best
            particle.best_position = particle.position.copy()
            particle.best_fitness = particle.fitness
            
            # Update global best
            if particle.fitness < self.global_best_fitness:
                self.global_best_fitness = particle.fitness
                self.global_best_position = particle.position.copy()
                self.global_best_schedule = particle.schedule.copy()
            
            self.particles.append(particle)
        
        print(f"✅ Swarm initialized: {self.swarm_size} particles")
        print(f"   Initial global best fitness: {self.global_best_fitness:.2f}")
    
    def _decode_position_to_schedule(self, position: np.ndarray) -> List[Dict]:
        """Decode continuous particle position to discrete berth schedule with FCFS constraints"""
        schedule = []
        berth_available_times = {berth.id: 0.0 for berth in self.berths}
        
        # Process vessels in FCFS order (by arrival time)
        for vessel_idx, vessel in enumerate(self.vessels_sorted):
            # Get berth preference from position
            berth_preference = position[vessel.id - 1]  # Vessel IDs start at 1
            
            # Find compatible berths sorted by preference
            compatible_berths = []
            for berth in self.berths:
                if (vessel.max_length <= berth.max_length and 
                    vessel.max_draft <= berth.max_draft):
                    # Calculate preference distance (lower is better)
                    distance = abs(berth_preference - berth.id)
                    compatible_berths.append((berth, distance))
            
            # Sort by preference (closest to preferred berth)
            compatible_berths.sort(key=lambda x: x[1])
            
            # Find best available berth
            best_berth = None
            best_start_time = float('inf')
            
            for berth, _ in compatible_berths:
                # Calculate earliest start time at this berth
                start_time = max(berth_available_times[berth.id], vessel.arrival_time)
                
                if start_time < best_start_time:
                    best_start_time = start_time
                    best_berth = berth
            
            if best_berth is not None:
                # Assign vessel to berth
                end_time = best_start_time + vessel.processing_time
                waiting_time = max(0, best_start_time - vessel.arrival_time)
                
                schedule.append({
                    'vessel_id': vessel.id,
                    'berth_id': best_berth.id,
                    'start_time': best_start_time,
                    'end_time': end_time,
                    'waiting_time': waiting_time,
                    'processing_time': vessel.processing_time,
                    'priority_score': vessel.priority_score
                })
                
                # Update berth availability
                berth_available_times[best_berth.id] = end_time
        
        return schedule
    
    def _calculate_fitness(self, schedule: List[Dict]) -> float:
        """Calculate fitness value for a schedule (lower is better)"""
        if not schedule:
            return float('inf')
        
        # Primary objective: total waiting time
        total_waiting_time = sum(assignment['waiting_time'] for assignment in schedule)
        
        # Secondary objective: makespan (completion time)
        makespan = max(assignment['end_time'] for assignment in schedule)
        
        # Weighted fitness function (can be tuned)
        fitness = 0.7 * total_waiting_time + 0.3 * makespan
        
        return fitness
    
    def _evaluate_particle(self, particle: Particle):
        """Evaluate particle fitness by decoding position to schedule"""
        # Decode position to schedule
        schedule = self._decode_position_to_schedule(particle.position)
        particle.schedule = schedule
        
        # Calculate fitness
        particle.fitness = self._calculate_fitness(schedule)
    
    def _update_particle_velocity(self, particle: Particle, iteration: int):
        """Update particle velocity using PSO equations"""
        # Adaptive inertia weight (linearly decreasing)
        w = self.inertia_weight * (1 - iteration / self.max_iterations) + 0.4
        
        # Random coefficients
        r1 = np.random.random(self.num_vessels)
        r2 = np.random.random(self.num_vessels)
        
        # PSO velocity update equation
        cognitive_component = self.cognitive_coeff * r1 * (particle.best_position - particle.position)
        social_component = self.social_coeff * r2 * (self.global_best_position - particle.position)
        
        particle.velocity = (w * particle.velocity + 
                           cognitive_component + 
                           social_component)
        
        # Velocity clamping to prevent explosion
        max_velocity = 2.0
        particle.velocity = np.clip(particle.velocity, -max_velocity, max_velocity)
    
    def _update_particle_position(self, particle: Particle):
        """Update particle position and handle boundary constraints"""
        # Update position
        particle.position = particle.position + particle.velocity
        
        # Handle boundary constraints [0, num_berths-1]
        particle.position = np.clip(particle.position, 0, self.num_berths - 0.01)
    
    def _calculate_swarm_diversity(self) -> float:
        """Calculate swarm diversity for convergence monitoring"""
        if len(self.particles) < 2:
            return 0.0
        
        # Calculate average distance between particles
        positions = np.array([p.position for p in self.particles])
        mean_position = np.mean(positions, axis=0)
        
        diversity = np.mean([np.linalg.norm(p.position - mean_position) for p in self.particles])
        return diversity
    
    def optimize(self) -> PSOSolution:
        """Run PSO optimization to find optimal berth schedule"""
        print("🚀 Starting PSO optimization for FCFS berth scheduling...")
        print(f"   Swarm size: {self.swarm_size} particles")
        print(f"   Max iterations: {self.max_iterations}")
        print(f"   Problem size: {self.num_vessels} vessels, {self.num_berths} berths")
        
        start_time = time.time()
        
        # Initialize swarm
        self._initialize_swarm()
        
        # Main PSO loop
        for iteration in range(self.max_iterations):
            # Update each particle
            for particle in self.particles:
                # Update velocity
                self._update_particle_velocity(particle, iteration)
                
                # Update position
                self._update_particle_position(particle)
                
                # Evaluate new position
                self._evaluate_particle(particle)
                
                # Update personal best
                if particle.fitness < particle.best_fitness:
                    particle.best_fitness = particle.fitness
                    particle.best_position = particle.position.copy()
                    
                    # Update global best
                    if particle.fitness < self.global_best_fitness:
                        self.global_best_fitness = particle.fitness
                        self.global_best_position = particle.position.copy()
                        self.global_best_schedule = particle.schedule.copy()
            
            # Track convergence
            self.convergence_history.append(self.global_best_fitness)
            self.diversity_history.append(self._calculate_swarm_diversity())
            
            # Progress reporting
            if (iteration + 1) % 10 == 0 or iteration == 0:
                print(f"   Iteration {iteration + 1:3d}: Best Fitness = {self.global_best_fitness:.2f}, "
                      f"Diversity = {self.diversity_history[-1]:.3f}")
        
        optimization_time = time.time() - start_time
        
        print(f"\n✅ PSO optimization completed in {optimization_time:.2f} seconds")
        print(f"   Final best fitness: {self.global_best_fitness:.2f}")
        print(f"   Convergence iterations: {len(self.convergence_history)}")
        
        # Create final solution
        self.solution = self._create_solution()
        
        return self.solution
    
    def _create_solution(self) -> PSOSolution:
        """Create final solution object from global best schedule"""
        if not self.global_best_schedule:
            return PSOSolution(
                schedule=[], total_waiting_time=0, makespan=0,
                berth_utilization={}, fcfs_compliance=0,
                convergence_iterations=0, final_fitness=float('inf')
            )
        
        # Calculate metrics
        total_waiting_time = sum(a['waiting_time'] for a in self.global_best_schedule)
        makespan = max(a['end_time'] for a in self.global_best_schedule)
        
        # Berth utilization
        berth_utilization = {}
        for berth in self.berths:
            berth_assignments = [a for a in self.global_best_schedule if a['berth_id'] == berth.id]
            if berth_assignments:
                total_processing = sum(a['end_time'] - a['start_time'] for a in berth_assignments)
                utilization = total_processing / makespan * 100
                berth_utilization[berth.id] = utilization
            else:
                berth_utilization[berth.id] = 0.0
        
        # FCFS compliance check
        fcfs_violations = 0
        for i, assignment in enumerate(self.global_best_schedule):
            if i > 0:
                prev_assignment = self.global_best_schedule[i-1]
                current_vessel = next(v for v in self.vessels if v.id == assignment['vessel_id'])
                prev_vessel = next(v for v in self.vessels if v.id == prev_assignment['vessel_id'])
                
                # Check FCFS violation
                if (current_vessel.arrival_time > prev_vessel.arrival_time and 
                    assignment['start_time'] < prev_assignment['end_time']):
                    fcfs_violations += 1
        
        fcfs_compliance = (len(self.global_best_schedule) - fcfs_violations) / len(self.global_best_schedule) * 100
        
        # Find convergence iteration (when fitness stopped improving significantly)
        convergence_iterations = len(self.convergence_history)
        if len(self.convergence_history) > 10:
            for i in range(len(self.convergence_history) - 10):
                if abs(self.convergence_history[i] - self.convergence_history[-1]) < 0.01:
                    convergence_iterations = i + 1
                    break
        
        return PSOSolution(
            schedule=self.global_best_schedule.copy(),
            total_waiting_time=total_waiting_time,
            makespan=makespan,
            berth_utilization=berth_utilization,
            fcfs_compliance=fcfs_compliance,
            convergence_iterations=convergence_iterations,
            final_fitness=self.global_best_fitness
        )
    
    def visualize_optimization_results(self):
        """Create comprehensive PSO optimization visualization"""
        if not self.solution:
            print("❌ No solution to visualize")
            return
        
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
        
        # 1. Convergence history
        iterations = range(1, len(self.convergence_history) + 1)
        ax1.plot(iterations, self.convergence_history, 'b-', linewidth=2, label='Best Fitness')
        ax1.set_xlabel('Iteration')
        ax1.set_ylabel('Fitness Value')
        ax1.set_title('PSO Convergence History', fontweight='bold')
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # Add convergence point marker
        ax1.axvline(x=self.solution.convergence_iterations, color='r', linestyle='--', 
                  alpha=0.7, label=f'Convergence at {self.solution.convergence_iterations}')
        ax1.legend()
        
        # 2. Swarm diversity
        ax2.plot(iterations, self.diversity_history, 'g-', linewidth=2, label='Swarm Diversity')
        ax2.set_xlabel('Iteration')
        ax2.set_ylabel('Diversity Measure')
        ax2.set_title('Swarm Diversity Evolution', fontweight='bold')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        # 3. Final schedule Gantt chart
        colors = plt.cm.Set3(np.linspace(0, 1, len(self.vessels)))
        
        for assignment in self.solution.schedule:
            color_idx = assignment['vessel_id'] - 1
            
            ax3.barh(assignment['berth_id'], assignment['end_time'] - assignment['start_time'],
                   left=assignment['start_time'], color=colors[color_idx], alpha=0.8,
                   edgecolor='black', linewidth=1)
            
            ax3.text(assignment['start_time'] + (assignment['end_time'] - assignment['start_time'])/2,
                    assignment['berth_id'], f'V{assignment["vessel_id"]}',
                    ha='center', va='center', fontweight='bold', fontsize=8)
        
        ax3.set_xlabel('Time (hours)')
        ax3.set_ylabel('Berth ID')
        ax3.set_title('PSO-Optimized Berth Schedule', fontweight='bold')
        ax3.grid(True, alpha=0.3)
        
        # 4. Berth utilization
        berth_ids = list(self.solution.berth_utilization.keys())
        utilizations = list(self.solution.berth_utilization.values())
        
        bars = ax4.bar(berth_ids, utilizations, color='lightblue', alpha=0.8,
                      edgecolor='black', linewidth=1)
        ax4.set_xlabel('Berth ID')
        ax4.set_ylabel('Utilization (%)')
        ax4.set_title('Berth Utilization', fontweight='bold')
        ax4.grid(True, alpha=0.3)
        
        # Add value labels on bars
        for bar, util in zip(bars, utilizations):
            height = bar.get_height()
            ax4.text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{util:.1f}%', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
    
    def print_detailed_results(self):
        """Print comprehensive PSO optimization results"""
        if not self.solution:
            print("❌ No solution available")
            return
        
        print("\n" + "="*80)
        print("🐝 PSO OPTIMIZATION RESULTS")
        print("="*80)
        
        print(f"\n⚙️  PSO Parameters:")
        print(f"   Swarm Size: {self.swarm_size} particles")
        print(f"   Max Iterations: {self.max_iterations}")
        print(f"   Inertia Weight: {self.inertia_weight}")
        print(f"   Cognitive Coefficient: {self.cognitive_coeff}")
        print(f"   Social Coefficient: {self.social_coeff}")
        
        print(f"\n📊 Optimization Performance:")
        print(f"   Final Fitness: {self.solution.final_fitness:.2f}")
        print(f"   Convergence Iterations: {self.solution.convergence_iterations}")
        print(f"   Total Waiting Time: {self.solution.total_waiting_time:.1f} hours")
        print(f"   Average Waiting Time: {self.solution.total_waiting_time/len(self.vessels):.1f} hours/vessel")
        print(f"   Makespan: {self.solution.makespan:.1f} hours")
        print(f"   FCFS Compliance: {self.solution.fcfs_compliance:.1f}%")
        
        print(f"\n📋 Optimized Schedule:")
        print(f"{'Vessel':<8} {'Berth':<8} {'Start':<8} {'End':<8} {'Wait':<8} {'Priority':<10}")
        print(f"{'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*10}")
        
        # Sort schedule by start time
        sorted_schedule = sorted(self.solution.schedule, key=lambda a: a['start_time'])
        
        for assignment in sorted_schedule:
            print(f"V{assignment['vessel_id']:<7} {assignment['berth_id']:<8} "
                  f"{assignment['start_time']:<8.1f} {assignment['end_time']:<8.1f} "
                  f"{assignment['waiting_time']:<8.1f} {assignment['priority_score']:<10.3f}")
        
        print(f"\n📈 Berth Utilization:")
        avg_utilization = np.mean(list(self.solution.berth_utilization.values()))
        for berth_id, utilization in self.solution.berth_utilization.items():
            print(f"   Berth {berth_id}: {utilization:.1f}% utilization")
        print(f"   Average Utilization: {avg_utilization:.1f}%")

print("✅ PSOFCFSScheduler class defined successfully")

In [ ]:
# Create the advanced example from the source material
print("🚢 Creating advanced vessel example for PSO optimization...")

# Define vessels for PSO demonstration (6 vessels, 4 berths)
vessels = [
    PSOVessel(
        id=1, arrival_time=0.0, processing_time=6.0, priority_score=0.85
    ),
    PSOVessel(
        id=2, arrival_time=1.0, processing_time=4.0, priority_score=0.65
    ),
    PSOVessel(
        id=3, arrival_time=2.0, processing_time=8.0, priority_score=0.90
    ),
    PSOVessel(
        id=4, arrival_time=2.5, processing_time=3.0, priority_score=0.45
    ),
    PSOVessel(
        id=5, arrival_time=4.0, processing_time=5.0, priority_score=0.70
    ),
    PSOVessel(
        id=6, arrival_time=4.0, processing_time=7.0, priority_score=0.80
    )
]

# Define berths with varying capabilities
berths = [
    PSOBerth(id=0, max_length=400, max_draft=16),  # Large berth
    PSOBerth(id=1, max_length=350, max_draft=14),  # Medium-large berth
    PSOBerth(id=2, max_length=300, max_draft=12),  # Medium berth
    PSOBerth(id=3, max_length=250, max_draft=10)   # Small berth
]

print(f"✅ Created {len(vessels)} vessels and {len(berths)} berths for PSO optimization")
print("\n📋 Vessel Details:")
for vessel in vessels:
    print(f"   Vessel {vessel.id}: Arrival={vessel.arrival_time}h, Process={vessel.processing_time}h, "
          f"Priority={vessel.priority_score:.2f}")

print("\n📋 Berth Details:")
for berth in berths:
    print(f"   Berth {berth.id}: Max Length={berth.max_length}m, Max Draft={berth.max_draft}m")

In [ ]:
# Initialize and run PSO optimization
print("🔧 Setting up PSO optimizer...")

# PSO parameters from source material
pso_params = {
    'swarm_size': 30,
    'max_iterations': 100,
    'inertia_weight': 0.7,
    'cognitive_coeff': 1.5,
    'social_coeff': 1.5,
    'random_seed': 42  # For reproducibility
}

# Create PSO scheduler
pso_scheduler = PSOFCFSScheduler(
    vessels=vessels,
    berths=berths,
    **pso_params
)

# Run optimization
solution = pso_scheduler.optimize()

print("\n✅ PSO optimization completed successfully!")

In [ ]:
# Display detailed results
pso_scheduler.print_detailed_results()

# Verify against expected results from source
print("\n" + "="*80)
print("🎯 VERIFICATION AGAINST EXPECTED RESULTS")
print("="*80)

expected_results = {
    'total_waiting_time': 6.0,
    'average_waiting_time': 1.0,
    'makespan': 14.0,
    'berth_utilization': 85.7,
    'convergence_iterations': 45,
    'final_fitness': 14.3
}

print("\n📊 Expected Performance Metrics (from source):")
for metric, expected_value in expected_results.items():
    if metric == 'berth_utilization':
        actual_value = np.mean(list(solution.berth_utilization.values()))
    elif metric == 'average_waiting_time':
        actual_value = solution.total_waiting_time / len(vessels)
    elif metric == 'final_fitness':
        actual_value = solution.final_fitness
    else:
        actual_value = getattr(solution, metric)
    
    tolerance = expected_value * 0.15  # 15% tolerance for metaheuristic
    accuracy = abs(actual_value - expected_value) <= tolerance
    print(f"   {metric.replace('_', ' ').title()}: Expected={expected_value}, "
          f"Actual={actual_value:.1f} {'✅' if accuracy else '❌'}")

# Expected vessel assignments from source
expected_assignments = [
    {'vessel_id': 1, 'berth_id': 0, 'start_time': 0.0, 'waiting_time': 0.0},
    {'vessel_id': 2, 'berth_id': 2, 'start_time': 1.0, 'waiting_time': 0.0},
    {'vessel_id': 3, 'berth_id': 0, 'start_time': 6.0, 'waiting_time': 4.0},
    {'vessel_id': 4, 'berth_id': 3, 'start_time': 2.5, 'waiting_time': 0.0},
    {'vessel_id': 5, 'berth_id': 2, 'start_time': 5.0, 'waiting_time': 2.0},
    {'vessel_id': 6, 'berth_id': 1, 'start_time': 4.0, 'waiting_time': 0.0}
]

print("\n📋 Expected Vessel Assignments (from source):")
for exp in expected_assignments:
    print(f"   Vessel {exp['vessel_id']} → Berth {exp['berth_id']}, "
          f"Start: {exp['start_time']}h, Wait: {exp['waiting_time']}h")

# Performance improvement verification
print(f"\n🚀 Performance Improvement Analysis:")
baseline_waiting = 9.5  # From Tier 2 results
pso_waiting = solution.total_waiting_time
improvement = (baseline_waiting - pso_waiting) / baseline_waiting * 100

print(f"   Baseline (Priority FCFS) Waiting: {baseline_waiting}h")
print(f"   PSO Optimized Waiting: {pso_waiting:.1f}h")
print(f"   Improvement: {improvement:.1f}% reduction")
print(f"   Expected improvement: 47% {'✅' if improvement >= 40 else '❌'}")

# FCFS compliance verification
print(f"\n📋 FCFS Compliance Verification:")
print(f"   FCFS Compliance Rate: {solution.fcfs_compliance:.1f}%")
print(f"   FCFS Maintained: {'✅' if solution.fcfs_compliance >= 90 else '❌'}")

# Berth utilization verification
avg_utilization = np.mean(list(solution.berth_utilization.values()))
print(f"\n📈 Berth Utilization Verification:")
print(f"   Average Utilization: {avg_utilization:.1f}%")
print(f"   Expected Utilization: 85.7% {'✅' if abs(avg_utilization - 85.7) <= 5 else '❌'}")

In [ ]:
# Visualize PSO optimization results
print("📊 Generating comprehensive PSO optimization visualization...")
pso_scheduler.visualize_optimization_results()

# Additional analysis: Particle behavior and swarm dynamics
print("\n📈 Additional Analysis: Swarm Dynamics and Particle Behavior")
print("="*70)

# Analyze convergence behavior
print(f"\n🔍 Convergence Analysis:")
if len(pso_scheduler.convergence_history) > 1:
    initial_fitness = pso_scheduler.convergence_history[0]
    final_fitness = pso_scheduler.convergence_history[-1]
    improvement = (initial_fitness - final_fitness) / initial_fitness * 100
    
    print(f"   Initial Best Fitness: {initial_fitness:.2f}")
    print(f"   Final Best Fitness: {final_fitness:.2f}")
    print(f"   Total Improvement: {improvement:.1f}%")
    print(f"   Convergence Point: Iteration {solution.convergence_iterations}")
    
    # Early vs late improvement analysis
    mid_point = len(pso_scheduler.convergence_history) // 2
    early_improvement = (pso_scheduler.convergence_history[0] - pso_scheduler.convergence_history[mid_point]) / pso_scheduler.convergence_history[0] * 100
    late_improvement = (pso_scheduler.convergence_history[mid_point] - pso_scheduler.convergence_history[-1]) / pso_scheduler.convergence_history[mid_point] * 100
    
    print(f"   Early Phase Improvement (first 50%): {early_improvement:.1f}%")
    print(f"   Late Phase Improvement (last 50%): {late_improvement:.1f}%")

# Swarm diversity analysis
print(f"\n🐝 Swarm Diversity Analysis:")
if pso_scheduler.diversity_history:
    initial_diversity = pso_scheduler.diversity_history[0]
    final_diversity = pso_scheduler.diversity_history[-1]
    diversity_reduction = (initial_diversity - final_diversity) / initial_diversity * 100
    
    print(f"   Initial Swarm Diversity: {initial_diversity:.3f}")
    print(f"   Final Swarm Diversity: {final_diversity:.3f}")
    print(f"   Diversity Reduction: {diversity_reduction:.1f}%")
    
    # Diversity stability check
    recent_diversity = pso_scheduler.diversity_history[-10:]
    diversity_std = np.std(recent_diversity)
    print(f"   Recent Diversity Stability (std): {diversity_std:.4f}")
    print(f"   Swarm Converged: {'✅' if diversity_std < 0.01 else '❌'}")

# Particle position analysis
print(f"\n📍 Particle Position Analysis:")
final_positions = np.array([p.position for p in pso_scheduler.particles])
global_best_pos = pso_scheduler.global_best_position

# Calculate distances from global best
distances_from_best = [np.linalg.norm(p.position - global_best_pos) for p in pso_scheduler.particles]
avg_distance = np.mean(distances_from_best)
max_distance = np.max(distances_from_best)

print(f"   Average Distance from Best: {avg_distance:.3f}")
print(f"   Maximum Distance from Best: {max_distance:.3f}")
print(f"   Position Convergence: {'✅' if avg_distance < 0.5 else '❌'}")

# Berth preference analysis
print(f"\n⚓ Berth Preference Analysis:")
berth_preferences = global_best_pos
for vessel_idx, preference in enumerate(berth_preferences):
    vessel = vessels[vessel_idx]
    assigned_berth = next((a['berth_id'] for a in solution.schedule if a['vessel_id'] == vessel.id), None)
    print(f"   Vessel {vessel.id}: Preference={preference:.2f}, Assigned=Berth {assigned_berth}, "
          f"Match={'✅' if abs(preference - assigned_berth) < 0.5 else '❌'}")

In [ ]:
# Sensitivity analysis: What if different PSO parameters?
print("🔍 SENSITIVITY ANALYSIS: PSO PARAMETERS")
print("="*50)

# Test different PSO parameter configurations
parameter_scenarios = [
    ("Balanced", 0.7, 1.5, 1.5),
    ("Exploration-Focused", 0.9, 1.0, 1.0),
    ("Exploitation-Focused", 0.4, 2.0, 2.0),
    ("Cognitive-Dominant", 0.7, 2.0, 1.0),
    ("Social-Dominant", 0.7, 1.0, 2.0)
]

results = []

for scenario_name, inertia, cognitive, social in parameter_scenarios:
    print(f"\n🧪 Testing {scenario_name} parameters...")
    
    # Create PSO scheduler with different parameters
    test_scheduler = PSOFCFSScheduler(
        vessels=vessels,
        berths=berths,
        swarm_size=20,  # Reduced for faster testing
        max_iterations=50,  # Reduced for faster testing
        inertia_weight=inertia,
        cognitive_coeff=cognitive,
        social_coeff=social,
        random_seed=42
    )
    
    # Run optimization
    test_solution = test_scheduler.optimize()
    
    # Calculate metrics
    total_wait = test_solution.total_waiting_time
    makespan = test_solution.makespan
    convergence_iter = test_solution.convergence_iterations
    final_fitness = test_solution.final_fitness
    
    results.append({
        'scenario': scenario_name,
        'inertia': inertia,
        'cognitive': cognitive,
        'social': social,
        'total_waiting': total_wait,
        'makespan': makespan,
        'convergence_iterations': convergence_iter,
        'final_fitness': final_fitness
    })

# Display sensitivity results
print("\n📊 PSO Parameter Sensitivity Results:")
print(f"{'Scenario':<20} {'Inertia':<10} {'Cogn':<8} {'Soc':<8} {'Wait':<8} {'Makespan':<10} {'Conv':<8} {'Fitness':<10}")
print(f"{'-'*20} {'-'*10} {'-'*8} {'-'*8} {'-'*8} {'-'*10} {'-'*8} {'-'*10}")

for result in results:
    print(f"{result['scenario']:<20} {result['inertia']:<10.1f} {result['cognitive']:<8.1f} "
          f"{result['social']:<8.1f} {result['total_waiting']:<8.1f} {result['makespan']:<10.1f} "
          f"{result['convergence_iterations']:<8} {result['final_fitness']:<10.1f}")

# Create sensitivity visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Waiting time comparison
scenario_names = [r['scenario'] for r in results]
waiting_times = [r['total_waiting'] for r in results]

bars1 = ax1.bar(range(len(scenario_names)), waiting_times, 
                color='lightcoral', alpha=0.8, edgecolor='black', linewidth=1)
ax1.set_xlabel('PSO Parameter Scenario')
ax1.set_ylabel('Total Waiting Time (hours)')
ax1.set_title('Sensitivity: Waiting Time Impact', fontweight='bold')
ax1.set_xticks(range(len(scenario_names)))
ax1.set_xticklabels(scenario_names, rotation=45, ha='right')
ax1.grid(True, alpha=0.3)

# Convergence iterations comparison
convergence_iters = [r['convergence_iterations'] for r in results]

bars2 = ax2.bar(range(len(scenario_names)), convergence_iters, 
                color='lightgreen', alpha=0.8, edgecolor='black', linewidth=1)
ax2.set_xlabel('PSO Parameter Scenario')
ax2.set_ylabel('Convergence Iterations')
ax2.set_title('Sensitivity: Convergence Speed', fontweight='bold')
ax2.set_xticks(range(len(scenario_names)))
ax2.set_xticklabels(scenario_names, rotation=45, ha='right')
ax2.grid(True, alpha=0.3)

# Makespan comparison
makespans = [r['makespan'] for r in results]

bars3 = ax3.bar(range(len(scenario_names)), makespans, 
                color='skyblue', alpha=0.8, edgecolor='black', linewidth=1)
ax3.set_xlabel('PSO Parameter Scenario')
ax3.set_ylabel('Makespan (hours)')
ax3.set_title('Sensitivity: Makespan Impact', fontweight='bold')
ax3.set_xticks(range(len(scenario_names)))
ax3.set_xticklabels(scenario_names, rotation=45, ha='right')
ax3.grid(True, alpha=0.3)

# Final fitness comparison
fitness_values = [r['final_fitness'] for r in results]

bars4 = ax4.bar(range(len(scenario_names)), fitness_values, 
                color='plum', alpha=0.8, edgecolor='black', linewidth=1)
ax4.set_xlabel('PSO Parameter Scenario')
ax4.set_ylabel('Final Fitness Value')
ax4.set_title('Sensitivity: Final Fitness', fontweight='bold')
ax4.set_xticks(range(len(scenario_names)))
ax4.set_xticklabels(scenario_names, rotation=45, ha='right')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🎯 Key Insights from PSO Parameter Sensitivity:")
print("   • Parameter settings significantly impact solution quality and convergence")
print("   • Balanced parameters provide consistent performance across metrics")
print("   • Exploration-focused settings may find better solutions but converge slower")
print("   • Exploitation-focused settings converge faster but may miss optimal solutions")
print("   • Cognitive vs social dominance affects swarm learning behavior")
print("   • Parameter tuning should consider both solution quality and computational time")

### Why this Tier exists vs Tiers 1-2
This Tier 3 addresses advanced optimization challenges beyond previous approaches:
- **Global optimization**: PSO explores entire solution space vs local greedy decisions
- **Swarm intelligence**: Collective learning vs individual vessel-by-vessel decisions
- **Adaptive search**: Dynamic balance between exploration and exploitation
- **Complex constraint handling**: Sophisticated FCFS constraint integration
- **Metaheuristic power**: Near-optimal solutions for complex, large-scale problems

### Pros / Cons vs Tiers 1-2
**Pros:**
- ✅ **Superior solution quality**: Typically 20-50% improvement over heuristic methods
- ✅ **Global optimization**: Avoids local optima traps that affect greedy approaches
- ✅ **Scalability**: Handles large problems efficiently with swarm parallelism
- ✅ **Adaptive learning**: Swarm intelligence discovers complex patterns
- ✅ **Constraint integration**: Sophisticated FCFS constraint handling
- ✅ **Convergence monitoring**: Clear termination criteria and progress tracking

**Cons:**
- ❌ **Parameter sensitivity**: Performance depends on PSO parameter tuning
- ❌ **Computational overhead**: More expensive than simple heuristics
- ❌ **No optimality guarantee**: Metaheuristic approach may not find true optimum
- ❌ **Stochastic results**: Different runs may produce different solutions
- ❌ **Complexity**: More sophisticated implementation and debugging

### When to use this Tier vs Tiers 1-2
- **Large-scale terminals** with 20+ vessels where heuristic performance is insufficient
- **Complex scheduling environments** with many interacting constraints
- **Performance-critical operations** where small improvements yield significant benefits
- **Strategic planning** where solution quality outweighs computational time
- **Benchmark studies** requiring state-of-the-art optimization performance
- **Research and development** of advanced scheduling systems

### Key innovations
- **Continuous-to-discrete encoding**: Innovative particle position decoding for berth scheduling
- **FCFS-aware fitness evaluation**: Constraint handling that preserves arrival order
- **Adaptive inertia weight**: Dynamic balance between exploration and exploitation
- **Swarm diversity monitoring**: Convergence detection and termination criteria
- **Multi-objective fitness**: Weighted combination of waiting time and makespan

### Performance expectations
- **Solution quality**: 20-50% improvement over heuristic approaches
- **Convergence**: Typically 40-60 iterations for medium-sized problems
- **Computational time**: Seconds to minutes for 50+ vessel problems
- **Scalability**: Linear to quadratic scaling with problem size
- **FCFS compliance**: 90-100% maintained while optimizing assignments
- **Consistency**: Similar results across multiple runs with proper parameter tuning